# 01 · Phase 1 — Level-vs-effect dissociation
**Retrieval as Treatment**

Runs both arms — arm 0 = parametric answer, arm 1 = same prompt + top-k BM25 passages — on **PopQA and TriviaQA**, then tests the paper's central claim:

> tau(x) = mu1(x) - mu0(x). Which arm does each signal know about, and does that
> knowledge actually drive a better retrieval decision than a single-proxy gate?

**What changed after the n=100 pilot**
- The headline is the **arm decomposition + cross-fitted tau evaluation**, not the harm map. Two earlier "effect" targets were circular and have been removed (see README). Confidence attaches to the parametric arm, retrieval scores to the retrieval arm; which one drives a better decision flips by dataset regime.
- Added **ambiguity features** (subject name length/tokens) — every genuine harm case in the pilot was a short polysemous name (`Idaho`, `Summit`, `X`).
- Added **normalized retrieval features**: raw BM25 score correlates r≈0.70 with name length, so it isn't comparable across queries.
- **Containment accuracy** is now the primary harm definition; token-F1 over-counted harm ~2× via partial credit. A harm audit labels every negative case genuine vs artifact.
- Map cells with n < 20 are suppressed; the go/no-go verdict now keys on the dissociation.
- **TriviaQA added.** PopQA's long tail is where retrieval nearly always helps (pilot: 63% of queries wrong in both arms, decision live on only 20%).

**Before you start:** Runtime → Change runtime type → **GPU**. Everything is written to Drive line by line; if the session dies, re-run from the top and it resumes.

In [ ]:
#@title 1 · Parameters + Drive + repo
REPO_URL = "https://github.com/<your-username>/retrieval-as-treatment"  #@param {type:"string"}
WORKDIR  = "/content/drive/MyDrive/research/retrieval-as-treatment"     #@param {type:"string"}
N_POPQA    = 300  #@param {type:"integer"}
N_TRIVIAQA = 300  #@param {type:"integer"}
# Smoke test at 300 + 300 first (~35 min on a T4). If the verdict is PROCEED,
# raise to 2000 + 1500 and re-run — nothing already done is recomputed.

from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
os.makedirs(WORKDIR, exist_ok=True)
if not os.path.exists('/content/rat'):
    subprocess.run(['git','clone',REPO_URL,'/content/rat'], check=True)
else:
    subprocess.run(['git','-C','/content/rat','pull'], check=True)
print('repo ready')

In [ ]:
#@title 2 · Deps (Java 21 for Pyserini + Python)  — ~4 min first time
!apt-get install -y -qq openjdk-21-jdk-headless > /dev/null 2>&1 || apt-get install -y -qq openjdk-21-jdk > /dev/null
!pip install -q -r /content/rat/requirements.txt
!pip install -q -e /content/rat
print('installed — if pip reports a torch/pillow conflict, Runtime → Restart, then continue from cell 3')

In [ ]:
#@title 3 · Environment
import os, glob, sys
jh = sorted(glob.glob('/usr/lib/jvm/java-21-openjdk*'))
assert jh, 'Java 21 missing — re-run cell 2'
os.environ['JAVA_HOME'] = jh[-1]
os.environ['PATH'] = jh[-1] + '/bin:' + os.environ['PATH']
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
!java -version 2>&1 | head -1

sys.path.insert(0, '/content/rat')
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

from rat.config import load_config, paths
from rat import phase1
cfg = load_config(WORKDIR, path='/content/rat/configs/phase1.yaml')
print(cfg.model_name, '| retriever:', cfg.retriever, '| top_k:', cfg.top_k)

In [ ]:
#@title 4 · Run Phase 1 on both datasets (resumable; loads the model once)
results = phase1.run_datasets(
    cfg,
    datasets=['popqa', 'triviaqa'],
    n_by_dataset={'popqa': N_POPQA, 'triviaqa': N_TRIVIAQA},
)

In [ ]:
#@title 5 · Cross-dataset summary — read this first
print(open(os.path.join(paths(cfg)['combined'], 'summary.md')).read())

In [ ]:
#@title 6 · Figures
from IPython.display import Image, display
import glob as _g
for ds in results:
    print('='*70); print(ds); print('='*70)
    for p in sorted(_g.glob(os.path.join(paths(cfg)['figures'], ds, '*.png'))):
        print(os.path.basename(p)); display(Image(p))

In [ ]:
#@title 7 · Full reports — copy these back
for ds, r in results.items():
    print('#'*70); print('#', ds); print('#'*70)
    print(r['report'])

In [ ]:
#@title 8 · (optional) Re-run analysis only, no GPU needed
# Use after editing rat/analysis.py: re-pulls the repo and rebuilds reports from cached logs.
# subprocess.run(['git','-C','/content/rat','pull'])
# import importlib, rat.analysis, rat.dissociation, rat.phase1
# for m in (rat.dissociation, rat.analysis, rat.phase1): importlib.reload(m)
# results = phase1.run_datasets(cfg, ['popqa','triviaqa'],
#                               n_by_dataset={'popqa': N_POPQA, 'triviaqa': N_TRIVIAQA},
#                               skip_generation=True)

## What to send back
1. `results/phase1_combined/summary.md`
2. `results/phase1/<dataset>/report.md` for both datasets
3. `results/phase1/<dataset>/features_binned.parquet` for both
4. `figures/phase1/<dataset>/arm_plane.png` and `tau_evaluation.png`

**The decision:** section 2 of each report gives a PROCEED / RE-CHECK verdict based on the dissociation statistic and whether a combined feature set beats the best single-proxy gate. Also check `harmed` in the cross-dataset table — if TriviaQA produces substantially more harm than PopQA, that confirms the dataset diagnosis from the pilot and TriviaQA becomes the primary dataset for Phase 2.